In [0]:

import yaml
from pathlib import Path

# Get the path relative to the current notebook location
current_dir = Path('.').resolve()
file_path = current_dir / '../../databricks.yml'

if file_path.exists():
    with open(file_path, 'r') as file:
        config = yaml.safe_load(file)
    
    # Get dev and prod configurations
    config = config.get('targets', {}).get('dev', {}).get('variables', {})
    # prod_config = config.get('targets', {}).get('prod', {}).get('variables', {})
    
    print("=" * 50)
    
    
    catalog=config.get('catalog')
    schema=config.get('schema')
    gold_schema=config.get('gold_schema')
    silver_schema=config.get('silver_schema')
    bronz_schema=config.get('bronz_schema')
                            
    print(f"catalog: {catalog}")
    print(f"schema: {schema}")
    print(f"gold_schema: {gold_schema}")
    print(f"silver_schema: {silver_schema}")
    print(f"bronz_schema: {bronz_schema }")
                            
    
    # print("\n" + "=" * 50)
    # print("PROD ENVIRONMENT CONFIGURATION")
    # print("=" * 50)
    # print(f"Catalog: {prod_config.get('catalog')}")
    # print(f"Schema: {prod_config.get('schema')}")
    # print(f"Gold Schema: {prod_config.get('gold_schema')}")
    # print(f"Silver Schema: {prod_config.get('silver_schema')}")
    # print(f"Bronze Schema: {prod_config.get('bronz_schema')}")
else:
    print(f"File not found: {file_path}. Please check the path and ensure the file exists.")

In [0]:
print(f"catalog: {catalog}")
print(f"schema: {schema}")
print(f"gold_schema: {gold_schema}")
print(f"silver_schema: {silver_schema}")
print(f"bronz_schema: {bronz_schema }")

Create Catalouge and Volume


In [0]:
# Create Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Create Schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronz_schema}")

# Create Volume for hospital landing zone
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.hospital_landing_zone")

In [0]:
# Create 'raw' directory in the volume using dbutils only if it does not exist
base_path = f'/Volumes/{catalog}/{schema}/hospital_landing_zone/raw'
subdirs = [base_path, f'{base_path}/fact', f'{base_path}/dimension']
created = []

for path in subdirs:
    try:
        dbutils.fs.ls(path)
    except Exception:
        dbutils.fs.mkdirs(path)
        created.append(path)

if created:
    print(f"Created directories: {', '.join(created)}")
else:
    print("All directories already exist. No new directories created.")